# Fase 1 — EDA & Temporal Split

Notebook ini **bukan sumber kebenaran**. Semua logic ada di `src/`; di sini hanya pemanggilan dan visualisasi.
Keputusan yang diambil tercatat di [`reports/decisions.md`](../reports/decisions.md).

Pertanyaan yang dijawab:
1. Berapa hari rentang data ini?
2. Apakah fraud rate stabil sepanjang waktu?
3. Kolom mana yang layak jadi node graph?

In [ ]:
import sys

sys.path.insert(0, "..")

import pandas as pd

from src.config import load_config, resolve_path
from src.data.eda import run_eda
from src.data.split import add_time_features, load_split_masks

pd.set_option("display.float_format", lambda x: f"{x:.4f}")

## 1. Rentang waktu dan split

In [ ]:
df = pd.read_parquet(
    resolve_path(load_config("data")["paths"]["merged_train"]),
    columns=["TransactionID", "TransactionDT", "isFraud"],
)
df = add_time_features(df)

print(f"rentang: hari {df['day'].min()}-{df['day'].max()} ({df['day'].max() + 1} hari)")
print(f"minggu: {df['week'].min()}-{df['week'].max()}")
print(f"fraud rate keseluruhan: {df['isFraud'].mean():.4f}")

In [ ]:
masks = load_split_masks()
summary = []
for name in ["is_train", "is_gap", "is_val", "is_test"]:
    m = masks[name].to_numpy()
    summary.append(
        {
            "split": name.replace("is_", ""),
            "n": int(m.sum()),
            "pct": m.mean(),
            "day_min": int(df.loc[m, "day"].min()),
            "day_max": int(df.loc[m, "day"].max()),
            "fraud_rate": df.loc[m, "isFraud"].mean(),
        }
    )
pd.DataFrame(summary)

## 2. Jalankan seluruh analisis EDA

Menghasilkan empat figur ke `reports/figures/`.

In [ ]:
results = run_eda()
list(results)

## 3. Fraud rate per minggu — apakah stabil?

**Tidak.** Lihat `reports/decisions.md` bagian T1.

In [ ]:
weekly = results["weekly"]
lo, hi = weekly.loc[weekly.fraud_rate.idxmin()], weekly.loc[weekly.fraud_rate.idxmax()]
print(f"terendah : minggu {int(lo.week)} = {lo.fraud_rate:.4f}")
print(f"tertinggi: minggu {int(hi.week)} = {hi.fraud_rate:.4f}")
print(f"rasio    : {hi.fraud_rate / lo.fraud_rate:.2f}x | std {weekly.fraud_rate.std():.4f}")
weekly

![weekly](../reports/figures/01_weekly_fraud_rate.png)

## 4. Profil per jam

Palung volume di hour 7-10, bukan 2-5 — `hour` bukan jam lokal pengguna (lihat T2).

In [ ]:
results["hourly"]

![hourly](../reports/figures/02_hourly_profile.png)

## 5. Kardinalitas kandidat node graph

`pct_singleton` menentukan: nilai yang muncul sekali menghasilkan node berderajat 1 yang tidak menghubungkan transaksi mana pun.

In [ ]:
results["cardinality"]

![cardinality](../reports/figures/03_cardinality.png)

## 6. Coverage entitas lintas periode

In [ ]:
results["coverage"]

![coverage](../reports/figures/04_entity_coverage.png)

## 7. Pola missing kolom V

Seluruh 339 kolom `Vxxx` jatuh ke dalam 15 blok dengan pola missing identik.

In [ ]:
import pyarrow.parquet as pq

from src.data.profiling import missing_pattern_groups

path = resolve_path(load_config("data")["paths"]["merged_train"])
v_cols = [c for c in pq.ParquetFile(path).schema.names if c.startswith("V")]
groups = missing_pattern_groups(pd.read_parquet(path, columns=v_cols), v_cols)

print(f"{len(v_cols)} kolom V -> {len(groups)} blok pola missing identik")
print(f"ukuran blok: {[len(g) for g in groups]}")